## Часть 4Д. Дерево решений изнутри

Дерево решений отвечает на цепочку вопросов вида **«признак ≤ порог?»**: «да» – идем в левую ветку, «нет» – в правую, пока не дойдем до **листа** с ответом.

1. **Мера беспорядка.** Насколько перемешаны классы в группе, показывает критерий Джини:
   $G = 1 - \sum_k p_k^2$, где $p_k$ – доля объектов класса $k$. Чистая группа дает $G = 0$, два класса поровну – $G = 0{,}5$.
2. **Качество вопроса.** Вопрос делит $n$ объектов на $n_L$ слева и $n_R$ справа. Беспорядок после вопроса – среднее с весами:
   $J = \dfrac{n_L}{n}G_L + \dfrac{n_R}{n}G_R$. Польза вопроса – **прирост** $\;\Delta = G_{\text{родитель}} - J$.
3. **Выбор вопроса.** Перебираются все признаки и все пороги; пороги – середины между соседними
   значениями признака: $t = \dfrac{v_i + v_{i+1}}{2}$. Берется вопрос с максимальным приростом.
4. **Рекурсия.** Для каждой из двух частей повторяется то же самое, пока узел не станет чистым,
   не кончится глубина `max_depth` или объектов не станет слишком мало. Ответ листа – **самый частый класс** в нем.

Алгоритм жадный: на каждом шаге выбирается лучший вопрос «здесь и сейчас», без заглядывания вперед.

> **Правило части.** Внутри функций дерева – только списки, словари, циклы и `math`, без numpy и sklearn.
> Sklearn нужен только для сверки в задании Д5.

In [ ]:
# Ячейка готова. Нужна, только если ноутбук запускается отдельно от ПР1.
# Если эта часть вставлена в ПР1 после части 4 – пропустите ее: все переменные уже есть.
import math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # в аудитории уберите эту строку, если графики не отображаются
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

RANDOM_STATE = 42
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='sort')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)
def accuracy(y_true, y_pred):   # та же функция, что в части 3
    correct = 0
    for t, p in zip(y_true, y_pred):
        if t == p:
            correct += 1
    return correct / len(y_true)

print('Обучающая:', X_train.shape, ' Тестовая:', X_test.shape)

In [ ]:
# Ячейка готова. Данные для этой части.
from sklearn.tree import export_text

# Учебный пример со слайдов: 10 учеников, признаки [часы подготовки, сон], 1 – сдал, 0 – не сдал
X_toy = [[1, 8], [2, 6], [2, 7], [3, 9], [4, 5], [5, 6], [6, 8], [7, 7], [8, 6], [9, 8]]
y_toy = [0, 0, 0, 1, 0, 1, 1, 1, 1, 1]
toy_features, toy_classes = ['часы', 'сон'], ['не сдал', 'сдал']

# Wine переводим в обычные списки. Масштабирование дереву не нужно – берем исходные признаки.
Xw_train, Xw_test = X_train.values.tolist(), X_test.values.tolist()
yw_train, yw_test = y_train.values.tolist(), y_test.values.tolist()
feature_names = list(data.feature_names)
class_names = ['сорт 0', 'сорт 1', 'сорт 2']
print('Учебный пример:', len(X_toy), 'объектов.  Wine:', len(Xw_train), 'обучающих объектов,', len(feature_names), 'признаков')

**Задание Д1. Мера беспорядка.** Напишите две функции:

1. `count_classes(y)` – словарь `{класс: сколько раз встречается}` (такой подсчет уже был в `majority_vote`);
2. `gini(y)` – критерий Джини $G = 1 - \sum_k p_k^2$. Для пустого списка верните `0.0`.

Критерий правильности – ожидаемые значения в выводе ячейки.

In [ ]:
# ЗАДАНИЕ Д1. Заполните пропуски.
def count_classes(y):
    counts = {}
    ...            # <-- пройдите по y и увеличьте счетчик каждой метки
    return counts

def gini(y):
    n = len(y)
    if n == 0:
        return 0.0
    ...            # <-- сумма квадратов долей по count_classes(y)
    return ...     # <-- 1 минус эта сумма

print('count_classes(y_toy) =', count_classes(y_toy), '  ожидается {0: 4, 1: 6}')
print(f'gini(y_toy)          = {gini(y_toy):.4f}   ожидается 0.4800')
print(f'gini([1]*10)         = {gini([1] * 10):.4f}   ожидается 0.0000')
print(f'gini([0, 1, 2])      = {gini([0, 1, 2]):.4f}   ожидается 0.6667')

**Вопрос Д1.** Чему равен $G$ корня дерева на Wine (три сорта в обучающей выборке)? Почему для трех классов
максимум $G$ больше, чем 0,5? Какой смысл у числа $G$, если метки в группе раздавать наугад в тех же пропорциях?

*Ваш ответ:*

**Задание Д2. Качество вопроса.**

1. `split(X, y, feature, threshold)` – делит выборку по вопросу `x[feature] <= threshold` и возвращает
   **четыре** списка: `X_left, y_left, X_right, y_right`;
2. `information_gain(y, y_left, y_right)` – прирост $\Delta = G(y) - \left(\dfrac{n_L}{n}G_L + \dfrac{n_R}{n}G_R\right)$.

Проверка на учебном примере: вопрос «часы ≤ 4,5» дает слева 4 «не сдал» и 1 «сдал», справа 5 «сдал»,
$J = 0{,}5 \cdot 0{,}32 + 0{,}5 \cdot 0 = 0{,}16$, прирост $0{,}48 - 0{,}16 = 0{,}32$.

In [ ]:
# ЗАДАНИЕ Д2. Заполните пропуски.
def split(X, y, feature, threshold):
    X_left, y_left, X_right, y_right = [], [], [], []
    for x, label in zip(X, y):
        ...        # <-- «да» – в левые списки, «нет» – в правые
    return X_left, y_left, X_right, y_right

def information_gain(y, y_left, y_right):
    n = len(y)
    J = ...        # <-- взвешенный Джини двух частей
    return ...     # <-- прирост

_, yl, _, yr = split(X_toy, y_toy, feature=0, threshold=4.5)
print('Слева:', yl, ' Справа:', yr)
print(f'Прирост «часы <= 4.5» = {information_gain(y_toy, yl, yr):.4f}   ожидается 0.3200')
_, yl, _, yr = split(X_toy, y_toy, feature=0, threshold=2.5)
print(f'Прирост «часы <= 2.5» = {information_gain(y_toy, yl, yr):.4f}   ожидается 0.3086')

**Вопрос Д2.** Вопрос «часы ≤ 1,5» отделяет одного ученика, и его группа идеально чистая ($G_L = 0$).
Почему прирост у такого вопроса маленький? Что изменилось бы, если бы мы усредняли $G_L$ и $G_R$ без весов $\frac{n_L}{n}$ и $\frac{n_R}{n}$?

*Ваш ответ:*

**Задание Д3. Лучший вопрос.**

1. `candidate_thresholds(X, feature)` – середины между соседними **уникальными отсортированными** значениями признака
   (пример: значения `[3, 1, 2, 2]` → `[1.5, 2.5]`; пригодятся `set` и `sorted`);
2. `best_split(X, y)` – перебирает все признаки (`range(len(X[0]))`) и все их пороги, возвращает
   `(лучший_признак, лучший_порог, лучший_прирост)`. Если прироста больше нуля нет – `(None, None, 0.0)`.

In [ ]:
# ЗАДАНИЕ Д3. Заполните пропуски.
def candidate_thresholds(X, feature):
    values = ...   # <-- уникальные отсортированные значения признака
    return ...     # <-- середины между соседними значениями

def best_split(X, y):
    best_feature, best_threshold, best_gain = None, None, 0.0
    for feature in range(len(X[0])):
        for threshold in candidate_thresholds(X, feature):
            ...    # <-- разбейте, посчитайте прирост, запомните лучший
    return best_feature, best_threshold, best_gain

print('candidate_thresholds([[3], [1], [2], [2]], 0) =', candidate_thresholds([[3], [1], [2], [2]], 0), '  ожидается [1.5, 2.5]')
f, t, g = best_split(X_toy, y_toy)
print(f'Учебный пример: «{toy_features[f]} <= {t}», прирост {g:.3f}   ожидается «часы <= 4.5», 0.320')
f, t, g = best_split(Xw_train, yw_train)
print(f'Корень Wine:    «{feature_names[f]} <= {t:.3f}», прирост {g:.3f}')

In [ ]:
# Как выглядит перебор: прирост для каждого порога лучшего признака в корне (ячейка готова)
f_root, t_root, _ = best_split(Xw_train, yw_train)
ts = candidate_thresholds(Xw_train, f_root)
gains = []
for t in ts:
    _, yl, _, yr = split(Xw_train, yw_train, f_root, t)
    gains.append(information_gain(yw_train, yl, yr))

plt.figure(figsize=(8, 3.8))
plt.plot(ts, gains, '.-')
plt.axvline(t_root, ls='--', c='gray', label=f'лучший порог {t_root:.3f}')
plt.xlabel(feature_names[f_root]); plt.ylabel('прирост'); plt.legend(); plt.grid(alpha=.3)
plt.title('Прирост для каждого порога в корне дерева')
plt.savefig('tree_gain.png', dpi=110, bbox_inches='tight'); plt.close()

n_questions = sum(len(candidate_thresholds(Xw_train, j)) for j in range(len(feature_names)))
print('Вопросов проверено в корне:', n_questions)

**Вопрос Д3.** Почему достаточно проверять только середины между соседними значениями, а не все числа подряд?
Сколько вопросов проверяет дерево в корне на Wine и во сколько раз это больше, чем на учебном примере (там их 12)?

*Ваш ответ:*

**Задание Д4. Дерево целиком.** Дерево храним во вложенных словарях:

```python
{'leaf': False, 'feature': 0, 'threshold': 4.5, 'gain': 0.32, 'n': 10, 'left': {...}, 'right': {...}}   # узел
{'leaf': True, 'class': 1, 'n': 5}                                                                     # лист
```

1. `most_common(y)` – самый частый класс (подсказка: `max(counts, key=counts.get)`);
2. `build_tree(X, y, depth, max_depth)` – рекурсивное построение:
   - вернуть лист, если `depth >= max_depth`, в узле меньше 2 объектов или в нем один класс;
   - найти `best_split`; если признак не найден – вернуть лист;
   - разбить выборку и построить левое и правое поддеревья с глубиной `depth + 1`;
3. `predict(tree, X)` – для каждого объекта спуститься от корня: `x[feature] <= threshold` – налево, иначе направо, пока не встретится лист.

In [ ]:
# ЗАДАНИЕ Д4. Заполните пропуски.
def most_common(y):
    counts = count_classes(y)
    return ...     # <-- класс с наибольшим счетчиком

def make_leaf(y):
    return {'leaf': True, 'class': most_common(y), 'n': len(y)}

def build_tree(X, y, depth=0, max_depth=3):
    if ...:        # <-- три условия остановки
        return make_leaf(y)
    feature, threshold, gain = ...   # <-- лучший вопрос
    if feature is None:
        return make_leaf(y)
    X_left, y_left, X_right, y_right = ...   # <-- разбиение
    return {'leaf': False, 'feature': feature, 'threshold': threshold, 'gain': gain, 'n': len(y),
            'left':  ...,    # <-- рекурсивный вызов для левой части
            'right': ...}    # <-- рекурсивный вызов для правой части

def predict(tree, X):
    preds = []
    for x in X:
        node = tree
        ...        # <-- спуск до листа
        preds.append(node['class'])
    return preds

In [ ]:
# Печать дерева и проверка на учебном примере (ячейка готова)
def print_tree(node, names, classes, indent=''):
    if node['leaf']:
        print(f"{indent}→ {classes[node['class']]}  (объектов: {node['n']})")
        return
    print(f"{indent}[{names[node['feature']]} ≤ {node['threshold']:.3f}?]  (объектов: {node['n']}, прирост: {node['gain']:.3f})")
    print(f'{indent}  да:');  print_tree(node['left'], names, classes, indent + '    ')
    print(f'{indent}  нет:'); print_tree(node['right'], names, classes, indent + '    ')

toy_tree = build_tree(X_toy, y_toy, max_depth=3)
print_tree(toy_tree, toy_features, toy_classes)
print('\nОжидается: корень «часы <= 4.5», слева вопрос «сон <= 8.5», справа лист «сдал» (5 объектов)')
print('Ученик [3 ч, 9 ч сна] ->', toy_classes[predict(toy_tree, [[3, 9]])[0]], '  ожидается: сдал')

**Задание Д5. Дерево на Wine и сверка с эталоном.** Обучите свое дерево глубины 3 на `Xw_train`, выведите его
и посчитайте точность на обучающей и тестовой выборках (функция `accuracy` из части 3).
Затем сравните с `DecisionTreeClassifier(max_depth=3)` – он строится тем же алгоритмом CART с критерием Джини.

In [ ]:
# ЗАДАНИЕ Д5. Заполните пропуски.
my_tree = ...          # <-- build_tree на Xw_train, yw_train с max_depth=3
print_tree(my_tree, feature_names, class_names)

my_pred = ...          # <-- предсказания на Xw_test
print('\nТочность на обучающей:', round(accuracy(yw_train, predict(my_tree, Xw_train)), 4))
print('Точность на тестовой: ', round(accuracy(yw_test, my_pred), 4))

In [ ]:
# Сверка со sklearn (ячейка готова)
sk_tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE).fit(X_train.values, y_train.values)
sk_pred = sk_tree.predict(X_test.values).tolist()

print(f"Корень sklearn: {feature_names[sk_tree.tree_.feature[0]]} ≤ {sk_tree.tree_.threshold[0]:.3f}")
print(f"Корень ваш:     {feature_names[my_tree['feature']]} ≤ {my_tree['threshold']:.3f}")
совпало = accuracy(sk_pred, my_pred)
print(f'Доля совпавших предсказаний: {совпало:.4f}')
if совпало < 1.0:
    print('\nЕсть расхождения. Дерево sklearn для сравнения по уровням:')
    print(export_text(sk_tree, feature_names=feature_names))

В отличие от kNN, полное совпадение со sklearn здесь не обязательно. Если в каком-то узле у двух вопросов
**одинаковый** прирост, ваша функция берет первый найденный, а sklearn перебирает признаки в случайном порядке.

**Вопрос Д4.** Если предсказания не совпали полностью – найдите узел, где деревья разошлись, и с помощью
`information_gain` проверьте, что приросты двух вопросов действительно равны. Если совпали – объясните, почему это не гарантировано.

**Вопрос Д5.** Выпишите из **своего** дерева правило для сорта 0: «Если ... ≤ ..., и ..., то сорт 0».
Почему дереву не понадобился `StandardScaler`, хотя kNN и логистической регрессии он был нужен?

*Ваши ответы:*

**Задание Д6. Глубина и переобучение.** Постройте зависимость точности своего дерева от `max_depth`
(от 1 до 8) на обучающей и тестовой выборках – так же, как для `k` в kNN и `C` в логистической регрессии.

In [ ]:
# ЗАДАНИЕ Д6. Заполните цикл.
depths = range(1, 9)
acc_tr_d, acc_te_d = [], []
for d in depths:
    t = ...                 # <-- дерево глубины d
    acc_tr_d.append(...)    # <-- точность на обучающей
    acc_te_d.append(...)    # <-- точность на тестовой

plt.figure(figsize=(8, 4))
plt.plot(depths, acc_tr_d, 'o-', label='обучающая')
plt.plot(depths, acc_te_d, 's-', label='тестовая')
plt.xlabel('max_depth'); plt.ylabel('точность'); plt.legend(); plt.grid(alpha=.3)
plt.title('Зависимость точности дерева от глубины')
plt.savefig('tree_depth.png', dpi=110, bbox_inches='tight'); plt.close()
for d, tr, te in zip(depths, acc_tr_d, acc_te_d):
    print(f'max_depth = {d}   обучающая {tr:.3f}   тестовая {te:.3f}')

**Вопрос Д6.** С какой глубины точность на обучающей выборке становится 100 %? Растет ли после этого
тестовая точность? Какой параметр kNN и какой параметр логистической регрессии играют ту же роль,
что `max_depth`, и в какую сторону их нужно менять, чтобы модель стала проще?

*Ваш ответ:*

---
### Задание повышенной сложности (к части 4Д)

1. Реализуйте энтропию $H = -\sum_k p_k \log_2 p_k$ и добавьте в `build_tree` параметр `criterion`.
   Совпадает ли дерево по энтропии с деревом по Джини? Сверьтесь с `DecisionTreeClassifier(criterion='entropy')`.
2. Добавьте параметр `min_samples_leaf`: вопрос допустим, только если в каждой части не меньше `min_samples_leaf` объектов.
   Как он влияет на график из задания Д6?
3. Важность признаков: для каждого признака сложите $\frac{n_{\text{узла}}}{n_{\text{всего}}}\cdot\Delta$ по всем узлам,
   где он используется, и нормируйте сумму к 1. Сравните с `sk_tree.feature_importances_`.
4. Ускорьте `best_split`: отсортируйте объекты по признаку один раз и пересчитывайте счетчики классов слева
   и справа при сдвиге порога, а не делите выборку заново. Замерьте ускорение на Wine.